# Cross-Sectional ML Market-Neutral Strategy
Leakage-aware expanding-window Ridge research on US sector ETFs. This notebook can run in Google Colab or inside the repository.

In [ ]:
!pip -q install yfinance scikit-learn


In [ ]:
from pathlib import Path
import os, sys, subprocess
if not Path('src/strategy.py').exists():
    subprocess.run(['git','clone','-q','https://github.com/Mohamed-Atigui/ml-market-neutral-strategy.git','repo'], check=True)
    os.chdir('repo')
sys.path.insert(0, str(Path.cwd()))
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
from src.strategy import mean_rank_ic, momentum_baseline_backtest, performance_metrics, walk_forward_backtest


## Data and features
The universe contains 11 US sector ETFs. Features are trailing momentum, realized volatility and drawdown measures computed only from information available at each month-end.

In [ ]:
UNIVERSE=['XLB','XLC','XLE','XLF','XLI','XLK','XLP','XLRE','XLU','XLV','XLY']
raw=yf.download(UNIVERSE,start='2017-01-01',auto_adjust=True,progress=False,threads=True)
prices=raw['Close'].dropna(how='all').ffill().dropna()
spy_raw=yf.download('SPY',start='2017-01-01',auto_adjust=True,progress=False)
spy=spy_raw['Close']
if isinstance(spy,pd.DataFrame): spy=spy.iloc[:,0]
spy=spy.reindex(prices.index).ffill().dropna()
prices=prices.reindex(spy.index).ffill().dropna()
prices.tail()


## Walk-forward Ridge model
At each month, the model is trained only on prior months. Predictions rank next-month relative returns; the portfolio is long the top three and short the bottom three with inverse-volatility sizing and 5 bps turnover costs.

In [ ]:
ml=walk_forward_backtest(prices,min_train_months=24,alpha=1.0,n_long=3,n_short=3,cost_bps=5.0)
strategy=ml['returns']
baseline=momentum_baseline_backtest(prices,strategy.index,cost_bps=5.0)
spy_monthly=spy.loc[spy.groupby(spy.index.to_period('M')).tail(1).index]
spy_forward=(spy_monthly.shift(-1)/spy_monthly-1.0).reindex(strategy.index).dropna()
common=strategy.index.intersection(spy_forward.index)
strategy=strategy.loc[common]; baseline=baseline.loc[common]; spy_forward=spy_forward.loc[common]
summary=pd.DataFrame({'ridge_ml_net':performance_metrics(strategy['net_return'],strategy['turnover']),'momentum_baseline_net':performance_metrics(baseline['net_return'],baseline['turnover']),'SPY':performance_metrics(spy_forward)})
summary.loc['mean_rank_ic','ridge_ml_net']=mean_rank_ic(ml['predictions'])
summary.round(4)


In [ ]:
returns=pd.DataFrame({'ridge_ml_net':strategy['net_return'],'momentum_baseline_net':baseline['net_return'],'SPY':spy_forward})
(1+returns).cumprod().plot(figsize=(11,5),title='Walk-forward Ridge strategy vs baselines')
plt.ylabel('Growth of $1'); plt.show()


## Chronological diagnostic

In [ ]:
periods=pd.DataFrame({'pre_2024':performance_metrics(strategy.loc[:'2023-12-31','net_return']),'2024_present':performance_metrics(strategy.loc['2024-01-01':,'net_return'])})
periods.round(4)


## Regularization sensitivity
A small diagnostic across Ridge penalties; this is not used to tune the reported test period.

In [ ]:
rows=[]
for alpha in [0.1,1.0,10.0,100.0]:
    trial=walk_forward_backtest(prices,min_train_months=24,alpha=alpha,n_long=3,n_short=3,cost_bps=5.0)
    r=trial['returns']['net_return'].reindex(common).dropna()
    rows.append({'alpha':alpha,**performance_metrics(r),'mean_rank_ic':mean_rank_ic(trial['predictions'])})
pd.DataFrame(rows)[['alpha','annualized_return','annualized_volatility','sharpe_ratio','max_drawdown','mean_rank_ic']].round(4)


## Interpretation
The goal is a transparent, reproducible research process rather than an over-tuned backtest: causal features, expanding-window validation, explicit costs, a market-neutral construction and simple benchmarks.